In [ ]:
import gensim.downloader

# Download 300-dimensional word2vec embeddings
model_name = 'word2vec-google-news-300'
n_features = 300

model = gensim.downloader.load(model_name)

# Load in transcript CSV file
import pickle

transcript_f = '../transcript/bronx_transcript.csv'
transcript_w2v = pd.read_csv(transcript_f)

# Convert words to lowercase
transcript_w2v['word'] = transcript_w2v.word.str.lower()

# Function to extract embeddings if available
def get_vector(word):
    if word in model.key_to_index:
        return model.get_vector(word, norm=True).astype(np.float32)
    return np.nan

# Extract embedding for each word
transcript_w2v['embedding'] = transcript_w2v.word.apply(get_vector)  
transcript_w2v = transcript_w2v.astype({'onset': 'float32', 'offset': 'float32'}, copy=False)

# Print out words not found in vocabulary
print(f'{(transcript_w2v.embedding.isna()).sum()} words not found:')
print(transcript_w2v.word[transcript_w2v.embedding.isna()].value_counts())

# Save transcript with embeddings using pickle
with open('../emb/bronx_w2v.pkl', 'wb') as f:
    pickle.dump(transcript_w2v, f)

transcript_f = '../emb/bronx_w2v.pkl'
if exists(transcript_f):
    with open(transcript_f, 'rb') as f:
        transcript_df = pickle.load(f)

In [ ]:
# 使う埋め込み (小文字で) : "gpt2", "word2vec", "gte", "llama"
EMBEDDINGS = ["w2v", "gpt2", "gte", "llama", "llama3", "gpt_oss"]   # 比較したいモデル群
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

def construct_predictors(transcript_df, n_features, stim_dur, tr=1.5):

    # Find total number of TRs
    stim_trs = np.ceil(stim_dur / tr)

    # Add column to transcript with TR indices
    transcript_df['TR'] = transcript_df.onset.divide(tr).apply(np.floor).apply(int)
    
    # Compile the words within each TR
    words_per_tr = transcript_df.groupby('TR')['word'].apply(list)
    
    # Average the embeddings within each TR
    embeddings_per_tr = transcript_df.groupby('TR')['embedding'].mean()
    
    # Loop through TRs
    words_trs = []
    embeddings_trs = []
    for t in np.arange(stim_trs):
        if t in words_per_tr:
            words_trs.append(words_per_tr[t])
    
            # Fill in empty TRs with zero vectors
            if embeddings_per_tr[t] is not np.nan:
                embeddings_trs.append(embeddings_per_tr[t])
            else:
                embeddings_trs.append(np.zeros(n_features))
        else:
            words_trs.append([])
            embeddings_trs.append(np.zeros(n_features))
    
    embeddings = np.vstack(embeddings_trs)
    return embeddings

def build_text_encoder(model_name, device="cpu"):
    """
    model_name: huggingface model id, e.g.
      - gpt2
      - Alibaba-NLP/gte-large
      - meta-llama/Llama-3.2-Text-Embedding  (例)
    """
    print("Loading model:", model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model

def encode_text_to_vector(tokenizer, model, text, device="cpu"):
    # 文字列を受け取り、最後層の hidden_state の平均を返す
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # 最終隠れ状態[-1] のトークン方向平均 -> 文ベクトル
    last_hidden = outputs.hidden_states[-1]  # (1, seq_len, dim)
    vec = last_hidden.mean(dim=1).squeeze().cpu().numpy()
    return vec

In [ ]:
from transformers import AutoTokenizer, AutoModel
import pickle
import os
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUT_DIR = "../emb"

# 単語リストを unique にして処理（同じ単語が多数回出るので効率化）
unique_words = sorted(set(transcript_df['word'].astype(str).values))
print("unique words:", len(unique_words))

# マップを作成するユーティリティ
def compute_and_attach_embeddings(transcript_df, tokenizer, model, device, model_tag):
    # compute embeddings for unique words
    word_to_vec = {}
    for w in tqdm(unique_words):
        try:
            v = encode_text_to_vector(tokenizer, model, w, device=device)
            word_to_vec[w] = v
        except Exception as e:
            print("Error encoding word:", w, e)
            word_to_vec[w] = np.zeros(model.config.hidden_size if hasattr(model.config,'hidden_size') else v.shape[0])
    # attach to a copy of transcript_df
    df = transcript_df.copy()
    df['embedding_'+model_tag] = df['word'].map(word_to_vec)
    return df, word_to_vec

# 実行例（モデル名は必要に応じて変更）
encoders = {}
word_maps = {}

# if "llama3" in EMBEDDINGS:
#     tok_llama3, m_llama3 = build_text_encoder("meta-llama/Llama-3.1-8B-Instruct", device=DEVICE)
#     df_llama3, map_llama3 = compute_and_attach_embeddings(transcript_df, tok_llama3, m_llama3, DEVICE, "llama3")
#     # 保存
#     df_llama3.to_pickle(os.path.join(OUT_DIR, "../emb/bronx_llama3.pkl"))
#     word_maps['llama3'] = map_llama3
#     print("Saved ../emb/bronx_llama3.pkl")

if "gpt_oss" in EMBEDDINGS:
    tok_gpt_oss, m_gpt_oss = build_text_encoder("openai/gpt-oss-20b", device=DEVICE)
    df_gpt_oss, map_gpt_oss = compute_and_attach_embeddings(transcript_df, tok_gpt_oss, m_gpt_oss, DEVICE, "gpt_oss")
    # 保存
    df_gpt_oss.to_pickle(os.path.join(OUT_DIR, "../emb/bronx_gpt_oss.pkl"))
    word_maps['gpt_oss'] = map_gpt_oss
    print("Saved ../emb/bronx_gpt_oss.pkl")

# if "gpt_oss" in EMBEDDINGS:
#     print("Loading gpt-oss-20b on CPU with MXFP4 quantization...")

#     from transformers import AutoTokenizer, AutoModel, Mxfp4Config

#     tok_gpt_oss = AutoTokenizer.from_pretrained(
#         "openai/gpt-oss-20b",
#         use_auth_token=True
#     )

#     quant_config = Mxfp4Config()

#     # ★ 重要：GPU を使わない
#     m_gpt_oss = AutoModel.from_pretrained(
#         "openai/gpt-oss-20b",
#         use_auth_token=True,
#         quantization_config=quant_config,
#         device_map={"": "cpu"},      # ★ 全て CPU に配置
#         low_cpu_mem_usage=True
#     )

#     df_gpt_oss, map_gpt_oss = compute_and_attach_embeddings(
#         transcript_df,
#         tok_gpt_oss,
#         m_gpt_oss,
#         device="cpu",                # ★ CPUで処理
#         model_tag="gpt_oss"
#     )

#     out_path = os.path.join(OUT_DIR, "../emb/bronx_gpt_oss.pkl")
#     df_gpt_oss.to_pickle(out_path)
#     word_maps["gpt_oss"] = map_gpt_oss

#     print(f"Saved {out_path}")


In [ ]:
from transformers import AutoTokenizer, AutoModel
import pickle
import os
OUT_DIR = "../output_csv"

# 単語リストを unique にして処理（同じ単語が多数回出るので効率化）
unique_words = sorted(set(transcript_df['word'].astype(str).values))
print("unique words:", len(unique_words))

# マップを作成するユーティリティ
def compute_and_attach_embeddings(transcript_df, tokenizer, model, device, model_tag):
    # compute embeddings for unique words
    word_to_vec = {}
    for w in tqdm(unique_words):
        try:
            v = encode_text_to_vector(tokenizer, model, w, device=device)
            word_to_vec[w] = v
        except Exception as e:
            print("Error encoding word:", w, e)
            word_to_vec[w] = np.zeros(model.config.hidden_size if hasattr(model.config,'hidden_size') else v.shape[0])
    # attach to a copy of transcript_df
    df = transcript_df.copy()
    df['embedding_'+model_tag] = df['word'].map(word_to_vec)
    return df, word_to_vec

# 実行例（モデル名は必要に応じて変更）
encoders = {}
word_maps = {}

# GPT-2
if "gpt2" in EMBEDDINGS:
    tok_gpt2, m_gpt2 = build_text_encoder("gpt2", device=DEVICE)
    df_gpt2, map_gpt2 = compute_and_attach_embeddings(transcript_df, tok_gpt2, m_gpt2, DEVICE, "gpt2")
    # 保存
    df_gpt2.to_pickle(os.path.join(OUT_DIR, "../emb/bronx_gpt2.pkl"))
    word_maps['gpt2'] = map_gpt2
    print("Saved ../emb/bronx_gpt2.pkl")

# # GTE
# if "gte" in EMBEDDINGS:
#     tok_gte, m_gte = build_text_encoder("Alibaba-NLP/gte-large", device=DEVICE)
#     df_gte, map_gte = compute_and_attach_embeddings(transcript_df, tok_gte, m_gte, DEVICE, "gte")
#     df_gte.to_pickle(os.path.join(OUT_DIR, "bronx_gte.pkl"))
#     word_maps['gte'] = map_gte
#     print("Saved bronx_gte.pkl")

if "gte" in EMBEDDINGS:
    print("Loading e5-large instead of gte-large")
    tok_gte, m_gte = build_text_encoder("intfloat/e5-large", device=DEVICE)
    df_gte, map_gte = compute_and_attach_embeddings(
        transcript_df,
        tok_gte,
        m_gte,
        DEVICE,
        "gte"    # 出力列名は embedding_gte になる
    )
    out_path = os.path.join(OUT_DIR, "../emb/bronx_gte.pkl")
    df_gte.to_pickle(out_path)
    word_maps['gte'] = map_gte
    print(f"Saved {out_path}")

# # LLaMA embedding (注意: モデル名は例)
# if "llama" in EMBEDDINGS:
#     # ここは実際の利用可能モデル名に置き換えてください。meta-llama/... は大きいです。
#     tok_llama, m_llama = build_text_encoder("NousResearch/Llama-2-7b-embedder", device=DEVICE)
#     df_llama, map_llama = compute_and_attach_embeddings(transcript_df, tok_llama, m_llama, DEVICE, "llama")
#     df_llama.to_pickle(os.path.join(OUT_DIR, "bronx_llama.pkl"))
#     word_maps['llama'] = map_llama
#     print("Saved bronx_llama.pkl")

# LLaMA-like embedding (use Mistral, Apache2.0, fully open)
if "llama" in EMBEDDINGS:
    model_id = "mistralai/Mistral-7B-v0.1"
    print(f"Loading LLaMA-like model (Mistral): {model_id}")

    tok_llama, m_llama = build_text_encoder(model_id, device=DEVICE)

    df_llama, map_llama = compute_and_attach_embeddings(
        transcript_df,
        tok_llama,
        m_llama,
        DEVICE,
        "llama"   # col name: embedding_llama
    )

    out_path = os.path.join(OUT_DIR, "../emb/bronx_llama.pkl")
    df_llama.to_pickle(out_path)
    word_maps['llama'] = map_llama
    print(f"Saved {out_path}")

# Word2Vec ベースは既に milkyway_w2v.pkl に embedding 列がある想定
if "w2v" in EMBEDDINGS:
    # transcript_df の 'embedding' 列が word2vec だと仮定して保存
    transcript_df.to_pickle(os.path.join(OUT_DIR, "../emb/transcript_w2v_from_upload.pkl"))
    print("Saved bronx_w2v_from_upload.pkl")